# 05 · Tier 2: graph neural networks

One unit per dataset × graph builder × seed. GraphSAGE and GAT on homogeneous graphs; an RGCN on the reduced
Home Credit heterogeneous graph (customer, bureau record, previous application). Neighbour sampling
(`NeighborLoader`) keeps memory flat on the 3.7M-node DGraph-Fin graph.

Pure GNNs are **not** expected to beat XGBoost on tabular credit data. They are here because (a) DGraph-Fin has a
published range to reproduce (SAGE 0.778, GAT ≈ 0.76, MLP 0.723 test AUC), and (b) their embeddings feed the
hybrid in notebook 06. Embeddings for train/val/test rows are saved as float16 `emb_<arch>.npy` with `emb_index.npy`.

Checkpoints: `ckpt/<arch>/last.pt` every epoch, `best.pt` at the best validation AUC. A killed job resumes.

**How this notebook runs.** Never by hand in an editor. `scripts/run_nb.sh 05_gnn dataset=... seed=...` executes it
with papermill on the GPU node, writes the executed copy to `runs/05_gnn/<dataset>/<graph_builder>/<seed>/executed.ipynb`,
and the last cell writes a `DONE` marker. If the process dies, the wrapper writes `FAILED` and the next `run_stage.sh`
picks the unit up again; checkpoints inside the run directory mean little work is repeated.

In [ ]:
# Parameters (papermill overrides these; keep names and types)
dataset = "dgraphfin"
seed = 0
graph_builder = "native"
run_dir = ""       # empty -> runs/<stage>/<dataset>/<graph_builder>/<seed>
force = 0          # 1 -> redo work even if checkpoints exist
quick = 0          # 1 -> subsampled data, few epochs; smoke test only

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import ckpt, config, paths, viz
from src.eval import results
from IPython.display import display

STAGE = "05_gnn"
warnings.filterwarnings("ignore")
seed = int(seed)
force, quick = bool(int(force)), bool(int(quick))
run_dir = Path(run_dir) if run_dir else paths.run_dir(STAGE, dataset, graph_builder, seed)
run_dir.mkdir(parents=True, exist_ok=True)
t0 = time.time()
print(f"stage={STAGE} dataset={dataset} graph_builder={graph_builder} seed={seed} quick={quick} force={force}")
print("run_dir:", run_dir)
pd.Series(paths.describe())

In [ ]:
from src.data import splits
from src.data.base import Bundle

cfg = config.load_config(dataset)
models_cfg = config.quick_override(cfg, quick)
bundle = Bundle.load(paths.processed_dir(dataset, quick))
split = splits.load_split(dataset, quick)
X, y = bundle.X, bundle.y.to_numpy()
tr, va, te = split["train"], split["val"], split["test"]
allowed = np.sort(np.concatenate([tr, va]))          # rows that may take part in graph construction
X_tr, X_va, X_te = X.iloc[tr], X.iloc[va], X.iloc[te]
y_tr, y_va, y_te = y[tr], y[va], y[te]
gdir = paths.graphs_dir(dataset, quick)
n_boot = 200 if quick else cfg["bootstrap"]["n_boot"]
print(bundle.summary())
print({p: len(split[p]) for p in splits.PARTS}, "| test positive rate:", round(float(np.nanmean(y_te)), 4))

## Build the PyG data object

In [ ]:
import torch

from src import device
from src.eval import metrics
from src.graphs import load_edges
from src.graphs import relational as REL
from src.models import gnn_archs as ARCH
from src.models import gnn_train as GT
from src.models.prep import TabularPrep

dev = device.torch_device()
gcfg = dict(models_cfg["gnn"])
print("device:", dev, "| gnn cfg:", gcfg)

prep = TabularPrep().fit(X_tr)
Z = prep.transform(X)
in_dims = num_rel = None
if graph_builder == "relational":
    payload = REL.load_hetero(gdir / "hetero_relational.npz")
    payload["x_customer"] = Z                     # same prepared customer features as every other model
    data, in_dims, num_rel = GT.make_hetero_data(payload, y)
    archs = [gcfg.get("hetero_arch", "rgcn")]
    print("hetero ->", {k: v for k, v in REL.hetero_stats(payload).items()})
else:
    edges = load_edges(dataset, graph_builder, cfg, quick, variant="full")
    data = GT.make_data(Z, y, edges)
    archs = list(gcfg["archs"])
print(data)
print("archs:", archs)

## Train

Loss on seed nodes only (`batch.batch_size` first rows), early stopping on validation AUC, `pos_weight` for the 1% positive rate on DGraph-Fin.

In [ ]:
preds, embs, histories, timings = {}, {}, {}, {}
lab_idx = np.concatenate([tr, va, te])
n_tr, n_va = len(tr), len(va)
pos_w = gcfg.get("pos_weight")

for arch in archs:
    ck = ckpt.TrainingCheckpoint(run_dir, tag=arch, every=gcfg.get("ckpt_every", 1))
    if force:
        shutil.rmtree(ck.dir, ignore_errors=True)
        ck = ckpt.TrainingCheckpoint(run_dir, tag=arch, every=gcfg.get("ckpt_every", 1))
    if arch == "rgcn":
        model = ARCH.build_model("rgcn", 0, gcfg, in_dims=in_dims, num_relations=num_rel)
    else:
        model = ARCH.build_model(arch, data.x.shape[1], gcfg)
    print(f"\n=== {arch}: {GT.count_params(model):,} parameters")
    t = time.time()
    hist = GT.train_gnn(model, data, tr, va, gcfg, ck, seed=seed, device=dev, pos_weight=pos_w)
    timings[arch] = round(time.time() - t, 1)
    logits, emb = GT.predict_gnn(model, data, lab_idx, gcfg, dev)
    p = GT.sigmoid(logits)
    preds[arch] = (p[n_tr:n_tr + n_va], p[n_tr + n_va:])
    np.save(run_dir / f"emb_{arch}.npy", emb.astype(np.float16))
    np.save(run_dir / f"logits_{arch}.npy", logits.astype(np.float32))
    histories[arch] = hist
    print(f"{arch}: val AUC {metrics.auc(y_va, preds[arch][0]):.4f}  test AUC {metrics.auc(y_te, preds[arch][1]):.4f}  "
          f"epochs {len(hist)}  {timings[arch]}s  emb {emb.shape}")
np.save(run_dir / "emb_index.npy", lab_idx)
for arch, hist in histories.items():
    fig = viz.plot_training_curves(hist, title=arch)
    viz.savefig(fig, run_dir, f"training_{arch}")
    plt.show()

## Test metrics and comparison with Tier 0 XGBoost

In [ ]:
rows = {arch: results.metrics_block(y_va, pv, y_te, pt) for arch, (pv, pt) in preds.items()}
table = pd.DataFrame({m: r["test"] for m, r in rows.items()}).T[["auc", "gini", "pr_auc", "ks", "capture_10", "brier", "ece"]]
if dataset == "dgraphfin":
    ref = cfg.get("reference_auc", {})
    table["paper_auc"] = [ref.get(a, np.nan) for a in table.index]
base = results.baseline_predictions(dataset, seed, "xgb", "test")
compare = results.compare_against(base, te, y_te, {a: pt for a, (pv, pt) in preds.items()}, n_boot=n_boot, seed=seed)
cmp_df = pd.DataFrame(compare)
fig = viz.plot_intervals(cmp_df[cmp_df.metric == "auc"], title=f"GNN test AUC minus XGBoost ({dataset}/{graph_builder}, seed {seed})")
viz.savefig(fig, run_dir, "auc_vs_xgb")
plt.show()
display(table.round(4))
cmp_df[["model", "metric", "base", "new", "diff", "ci_low", "ci_high", "significant"]].round(4)

## Curves and a look at the embedding

2-D PCA of the validation embeddings coloured by label. Separation here is what the hybrid will exploit.

In [ ]:
from sklearn.decomposition import PCA

_, p_base, _ = results.align(base, pd.DataFrame({"position": te, "y": y_te, "p": preds[archs[0]][1]}))
fig = viz.plot_roc_pr(y_te, {"xgb (tier 0)": p_base, **{a: pt for a, (pv, pt) in preds.items()}}, title=f"{dataset} test")
viz.savefig(fig, run_dir, "roc_pr")
plt.show()

arch0 = archs[0]
E = np.load(run_dir / f"emb_{arch0}.npy").astype(np.float32)[n_tr:n_tr + n_va]
rng = np.random.default_rng(seed)
samp = rng.choice(len(E), min(20000, len(E)), replace=False)
pc = PCA(2, random_state=seed).fit_transform(E[samp])
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(pc[y_va[samp] == 0, 0], pc[y_va[samp] == 0, 1], s=2, alpha=0.3, label="good")
ax.scatter(pc[y_va[samp] == 1, 0], pc[y_va[samp] == 1, 1], s=4, alpha=0.6, c="C3", label="bad")
ax.legend(); ax.set(title=f"{arch0} validation embedding, PCA-2")
viz.savefig(fig, run_dir, f"embedding_{arch0}")
plt.show()

In [ ]:
for arch, (pv, pt) in preds.items():
    results.save_predictions(run_dir, arch, {"val": (va, y_va, pv), "test": (te, y_te, pt)})
results.save_metrics(run_dir, {"models": rows, "compare": compare,
                               "params": {"graph_builder": graph_builder, "seed": seed, "quick": quick, "gnn_cfg": gcfg, "archs": archs,
                                          "timings_s": timings, "epochs": {a: len(h) for a, h in histories.items()},
                                          "n_edges": int(data.edge_index.shape[1]), "device": str(dev)}})
summary = {"test_auc": {a: round(r["test"]["auc"], 4) for a, r in rows.items()}, "timings_s": timings, "elapsed_s": round(time.time() - t0, 1)}
ckpt.mark_done(run_dir, summary)
summary